# **Essential Component of Intelligent Agent**

### **Utility Function**

In [15]:
# The code below shows a simple utility function for travel agent

import random 

def travel_utility_function(travel_option):
    """
        A Utility function that evaluate travel option based on
        price, comfort and convinience
    """

    price_utility = (1000 - travel_option['price']) * 0.05
    comfort_utility = travel_option['comfort_rating'] * 10
    convenience_utility = travel_option['convenience_score'] * 15

    total_utility = price_utility + comfort_utility + convenience_utility

    return total_utility


# Define some example travel options 
travel_options = [
    {
        'name': 'Budget Airline',
        'price': 300,
        'comfort_rating': 3,
        'convenience_score': 2
    },
    {
        'name': 'Premium Airline',
        'price': 800,
        'comfort_rating': 8,
        'convenience_score': 7
    },
    {
        'name': 'Train',
        'price': 200,
        'comfort_rating': 6,
        'convenience_score': 5
    },
    {
        'name': 'Road Trip',
        'price': 150,
        'comfort_rating': 4,
        'convenience_score': 3
    }
]


# Calculate and print utility for each travel option
for option in travel_options:
    utility = travel_utility_function(option)
    print(f"Option Name :- {option['name']}")
    print(f"Price: ${option['price']}, Comfort: {option['comfort_rating']}/10, Convenience: {option['convenience_score']}/10")
    print(f"Utility: {utility:.2f}\n")


# Fin the best option based on Utility
best_option = max(travel_options , key = travel_utility_function)
print(f"The best travel option according to our utility function is: {best_option['name']}")
print(f"Its utility value is: {travel_utility_function(best_option):.2f}")

Option Name :- Budget Airline
Price: $300, Comfort: 3/10, Convenience: 2/10
Utility: 95.00

Option Name :- Premium Airline
Price: $800, Comfort: 8/10, Convenience: 7/10
Utility: 195.00

Option Name :- Train
Price: $200, Comfort: 6/10, Convenience: 5/10
Utility: 175.00

Option Name :- Road Trip
Price: $150, Comfort: 4/10, Convenience: 3/10
Utility: 127.50

The best travel option according to our utility function is: Premium Airline
Its utility value is: 195.00


# **Enhancing Agent Capabilities using Using Generative AI**

In [16]:
import getpass 
api_key = getpass.getpass(prompt="Enter Openai Api key :-  " )

In [17]:
def book_flight(passenger_name : str , from_city : str , to_city : str , travel_date : str):
    # Simply Returns a String 
    return {"response": f"A {travel_date} flight has been booked from {from_city} to {to_city} for {passenger_name}"}

In [18]:
from openai import OpenAI
import json 

openai = OpenAI(api_key=api_key)

tools = [{
        "type": "function",
        "function":{
            "name": "book_flight",
            "description": "Book a flight for the customer. Call this whenever you need to book a flight, for example when a customer asks 'I want to book a flight from Los Angeles to New York'",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_city": {
                        "type": "string",
                        "description": "The departure city.",
                    },
                    "to_city": {
                        "type": "string",
                        "description": "The arrival city.",
                    },
                    "travel_date": {
                        "type": "string",
                        "description": "The date of travel.",
                    },
                    "passenger_name": {
                        "type": "string",
                        "description": "The passenger's legal name.",
                    },
                },
                "required": ["passenger_name","from_city", "to_city", "travel_date"],
                "additionalProperties": False,
            }
    }
}]


# This is the main Function that Call the LLM

def travel_agent(user_message : str , messages : list) -> str:
    messages.append({'role' : 'user' , 'content' : user_message})
    try:
        response = openai.chat.completions.create(
            model = 'gpt-4o-mini',
            messages = messages ,
            tools=tools
        )
        if response.choices[0].message.content:
            return response.choices[0].message.content
        elif response.choices[0].message.tool_calls:
            tool_call = response.choices[0].message.tool_calls[0]
            arguments = json.loads(tool_call.function.arguments)
            from_city = arguments.get('from_city')
            to_city = arguments.get('to_city')
            travel_date = arguments.get('travel_date')
            passenger_name = arguments.get('passenger_name')

            # Call Travel Booking function
            booking_cofirmation = book_flight(passenger_name=passenger_name, from_city=from_city, to_city=to_city, travel_date=travel_date)

            function_call_result_message = {
                "role" : "tool" ,
                "content" : json.dumps({
                    "confirmation_message" : booking_cofirmation
                }),
                "tool_call_id" : response.choices[0].message.tool_calls[0].id
            }

            messages.append(response.choices[0].message)
            messages.append(function_call_result_message)
            response = openai.chat.completions.create(model = 'gpt-4o-mini',messages=messages)

            return response.choices[0].message.content 
    except Exception as e:
        return f"Error Occured : {str(e)}"


In [ ]:
import ipywidgets as widgets 
from IPython.display import display

messages = [
    {"role": "system", "content": """You are a helpful travel agent assistant. 
     Use the supplied tools to assist the customer. 
     If you don't have enough information to book, just ask. 
     When you have the travel cities, dates and name, you can use the tool to book the ticket."""},
]

def on_submit(b):
    message = text_input.value
    output.value = output.value.replace("</div>", f"<p style='border: 1px solid gray; border-radius: 5px; margin-bottom: 4px; padding: 3px'><b style'color: blue;'>User:</b> {message}</p></div>")
    text_input.value = ''
    
    response = travel_agent(user_message=message, messages=messages)
    messages.append({"role": "assistant", "content": response})
    output.value = output.value.replace("</div>", f"<p style='border: 1px solid gray; border-radius: 5px; margin-bottom: 4px; padding: 3px'><b style'color: red;'>Travel agent:</b> {response}</p></div>")

text_input = widgets.Text(description="You:")
output = widgets.HTML(value="<div style='border: 2px solid gray; padding: 10px; width: 500px; height: 400px;'>YOUR AI TRAVEL AGENT</div>")
button = widgets.Button(description="Send")
button.on_click(on_submit)
text_input.on_submit(lambda x: on_submit(None))

display(output, widgets.HBox([text_input, button]))

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4552\2325586477.py:24: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(lambda x: on_submit(None))


HTML(value="<div style='border: 2px solid gray; padding: 10px; width: 500px; height: 400px;'>YOUR AI TRAVEL AG…

In [20]:
print(messages)

[{'role': 'system', 'content': "You are a helpful travel agent assistant. \n     Use the supplied tools to assist the customer. \n     If you don't have enough information to book, just ask. \n     When you have the travel cities, dates and name, you can use the tool to book the ticket."}, {'role': 'user', 'content': 'I want to book a flight from Los Angeles to New York'}, {'role': 'assistant', 'content': 'Could you please provide me with the date of travel and your legal name for the booking?'}, {'role': 'user', 'content': 'Name : Paras and date : 18-06-2026'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_fFvyocuBedVHBGmHFRGoVnyz', function=Function(arguments='{"from_city":"Los Angeles","to_city":"New York","travel_date":"18-06-2026","passenger_name":"Paras"}', name='book_flight'), type='function')]), {'role': 'tool', 'content': '{"confirmation_message": {"